# Peak correlation analysis

This notebook finds peaks whose intensities **co-vary across samples**.
Because peaks are identified independently in each sample, the same physical
signal can appear at slightly different m/z values. We therefore first **bin
peaks by m/z** (gap-based clustering with a ppm tolerance), then compute
pairwise Pearson correlations on the resulting intensity vectors.

Two workflows are provided:

| Section               | Approach                                                                                       |
| --------------------- | ---------------------------------------------------------------------------------------------- |
| **Reference peak**    | Pick one peak (by m/z, formula, or compound) and find the top-N signals that correlate with it |
| **Global clustering** | Compute the full correlation matrix and group co-varying peaks with hierarchical clustering    |


### Load peaks


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from mascope_sdk import MascopeClient


pio.templates.default = "plotly_dark"  # or "plotly_white"

mascope = MascopeClient(workspace="My Workspace")

run_global_clustering = False  # set True to run Section 2 (can be slow)

# Load all peaks across batches (adjust dataset/batches to your data)
peaks = mascope.load_peaks(dataset="My Dataset", batches="My Batch")

print(
    f"{len(peaks)} rows, {peaks['sample_item_name'].nunique()} samples, "
    f"{peaks['peak_id'].nunique()} unique peak_ids"
)
peaks.head()

### m/z binning

Each sample identifies peaks independently, so the same physical signal has
slightly different m/z across samples. We cluster peaks into **m/z bins** using
a gap-based algorithm:

1. Sort all peaks by m/z
2. Walk through the sorted list; start a new bin whenever the gap to the next
   peak exceeds a ppm-based tolerance
3. Assign each peak its bin's **representative m/z** (intensity-weighted mean)

This avoids fixed-width bin boundary artefacts.


In [ ]:
# --- Parameters ---
mz_tolerance_ppm = 2  # max gap (in ppm) within a bin
min_sample_fraction = 0.5  # bin must appear in at least this fraction of samples
min_median_height = 0  # minimum median height across samples to keep a bin


def bin_peaks_by_mz(df: pd.DataFrame, tol_ppm: float) -> pd.DataFrame:
    """Assign an `mz_bin` label to each peak using gap-based clustering."""
    # For correlation we need one value per (sample, mz_bin).
    # If a peak is expanded into multiple match rows, collapse to one row
    # per (sample, peak) first - keep the original mz and height.
    unique_peaks = df.drop_duplicates(subset=["sample_item_id", "peak_id"]).copy()

    sorted_df = unique_peaks.sort_values("mz").reset_index(drop=True)
    mz = sorted_df["mz"].values

    # Compute ppm gaps between consecutive peaks
    gaps_ppm = np.diff(mz) / mz[:-1] * 1e6

    # Assign bin ids: increment whenever gap exceeds tolerance
    bin_ids = np.zeros(len(mz), dtype=int)
    bin_ids[1:] = np.cumsum(gaps_ppm > tol_ppm)

    sorted_df["mz_bin"] = bin_ids
    return sorted_df


binned = bin_peaks_by_mz(peaks, mz_tolerance_ppm)

# Compute representative m/z per bin (intensity-weighted mean)
_wt = binned["mz"] * binned["height"]
bin_mz = (
    _wt.groupby(binned["mz_bin"]).sum()
    / binned["height"].groupby(binned["mz_bin"]).sum()
)
bin_mz.name = "mz_representative"
binned = binned.merge(bin_mz, on="mz_bin")

# Attach the best match label (if any) to each bin for display
if "target_compound_formula" in binned.columns:
    bin_labels = (
        binned.dropna(subset=["target_compound_formula"])
        .drop_duplicates(subset=["mz_bin"])
        .set_index("mz_bin")[["target_compound_formula", "target_ion_formula"]]
    )
else:
    bin_labels = pd.DataFrame()

n_bins = binned["mz_bin"].nunique()
print(f"{n_bins} m/z bins from {len(binned)} peaks")

### Build the intensity matrix

Pivot to a matrix where **rows = samples** and **columns = m/z bins**,
values = peak height. If a sample has multiple peaks in one bin, they are
summed. Missing values (peak not detected in a sample) are left as `NaN`.


In [ ]:
intensity_matrix = binned.pivot_table(
    index="sample_item_id",
    columns="mz_bin",
    values="height",
    aggfunc="sum",
)

n_samples = len(intensity_matrix)

# Filter bins: must appear in enough samples and have sufficient intensity
presence = intensity_matrix.notna().sum() / n_samples
median_height = intensity_matrix.median()
keep = (presence >= min_sample_fraction) & (median_height >= min_median_height)
intensity_matrix = intensity_matrix.loc[:, keep]

print(
    f"Intensity matrix: {intensity_matrix.shape[0]} samples × "
    f"{intensity_matrix.shape[1]} bins (after filtering)"
)

if n_samples < 10:
    print(
        f"⚠ Only {n_samples} samples, correlations may not be reliable. "
        "Consider loading more batches."
    )

intensity_matrix.head()

---

## Section 1: Reference-peak correlation

Pick a reference peak and find the signals that correlate most strongly with it
across samples.


In [ ]:
# --- Choose a reference peak ---
# Option A: by m/z (finds the nearest bin)
reference_mz = 123.4567  # <- change to an m/z of interest in your data

# Option B: by compound or ion formula (uncomment one)
# reference_mz = bin_labels[bin_labels["target_compound_formula"] == "CH4N2O"].index[0]
# reference_mz = bin_labels[bin_labels["target_ion_formula"] == "CH5N2O+"].index[0]

# Minimum |r| to consider a signal correlated (or anti-correlated)
correlation_threshold = 0.7
# Minimum number of overlapping samples for a valid correlation
min_periods = max(5, n_samples // 3)

In [ ]:
# Resolve reference_mz to the nearest bin
if isinstance(reference_mz, (int, float)):
    ref_bin = (bin_mz - reference_mz).abs().idxmin()
else:
    ref_bin = reference_mz  # already a bin id

ref_label = f"m/z {bin_mz[ref_bin]:.4f}"
if ref_bin in bin_labels.index:
    ref_label += f" ({bin_labels.loc[ref_bin, 'target_ion_formula']})"
print(f"Reference bin: {ref_label}")

if ref_bin not in intensity_matrix.columns:
    raise ValueError(
        f"Reference bin {ref_bin} was removed by filtering. "
        "Try lowering min_sample_fraction or min_median_height."
    )

# Correlate reference against all other bins
ref_series = intensity_matrix[ref_bin]
correlations = intensity_matrix.apply(
    lambda col: col.corr(ref_series, min_periods=min_periods)
)
correlations = correlations.drop(ref_bin, errors="ignore").dropna()

# Split into positively and negatively correlated
pos_corr = correlations[correlations >= correlation_threshold].sort_values(
    ascending=False
)
neg_corr = correlations[correlations <= -correlation_threshold].sort_values(
    ascending=True
)

print(f"{len(pos_corr)} positively correlated bins (r ≥ {correlation_threshold})")
print(f"{len(neg_corr)} anti-correlated bins (r ≤ -{correlation_threshold})")

# Build a combined results table
thresholded = pd.concat([pos_corr, neg_corr]).reset_index()
thresholded.columns = ["mz_bin", "correlation"]
thresholded["mz"] = thresholded["mz_bin"].map(bin_mz)

thresholded["direction"] = np.where(
    thresholded["correlation"] > 0, "positive", "negative"
)
if not bin_labels.empty:
    thresholded = thresholded.merge(
        bin_labels, left_on="mz_bin", right_index=True, how="left"
    )
thresholded.sort_values("correlation", ascending=False)

In [ ]:
# Bar chart of correlated signals
thresholded["label"] = thresholded["mz"].apply(lambda v: f"{v:.4f}")
if "target_ion_formula" in thresholded.columns:
    has_formula = thresholded["target_ion_formula"].notna()
    thresholded.loc[has_formula, "label"] = (
        thresholded.loc[has_formula, "label"]
        + " ("
        + thresholded.loc[has_formula, "target_ion_formula"]
        + ")"
    )

fig = px.bar(
    thresholded.sort_values("correlation"),
    x="correlation",
    y="label",
    color="direction",
    color_discrete_map={"positive": "#636EFA", "negative": "#EF553B"},
    orientation="h",
    title=f"Signals correlated with {ref_label} (|r| ≥ {correlation_threshold})",
    hover_data=["mz"],
)
fig.update_layout(
    yaxis=dict(categoryorder="total ascending", title="m/z bin"),
    xaxis_title="Pearson r",
)
fig.show()

In [ ]:
# Timeseries: reference vs correlated signals
plot_bins = [ref_bin] + thresholded["mz_bin"].tolist()
plot_df = intensity_matrix[plot_bins].copy()

# Add sample datetime for x-axis
sample_dates = binned.drop_duplicates(subset=["sample_item_id"]).set_index(
    "sample_item_id"
)["datetime_utc"]
plot_df = plot_df.join(sample_dates).melt(
    id_vars="datetime_utc", var_name="mz_bin", value_name="height"
)


# Build readable labels
def _bin_label(b):
    lbl = f"m/z {bin_mz[b]:.4f}"
    if b in bin_labels.index:
        lbl += f" ({bin_labels.loc[b, 'target_ion_formula']})"
    if b == ref_bin:
        lbl += " [ref]"
    return lbl


plot_df["label"] = plot_df["mz_bin"].map(_bin_label)

# Add correlation coefficient and direction for hover / marker shape
corr_map = thresholded.set_index("mz_bin")["correlation"]
dir_map = thresholded.set_index("mz_bin")["direction"]
plot_df["r"] = plot_df["mz_bin"].map(corr_map).round(3).fillna("reference")
plot_df["direction"] = plot_df["mz_bin"].map(dir_map).fillna("reference")

fig = px.scatter(
    plot_df.dropna(subset=["height"]),
    x="datetime_utc",
    y="height",
    color="label",
    symbol="direction",
    symbol_map={"reference": "circle", "positive": "circle", "negative": "diamond"},
    hover_data=["r"],
    title=f"Reference peak vs correlated signals (|r| ≥ {correlation_threshold})",
)
fig.update_layout(
    xaxis_title="Datetime (UTC)",
    yaxis_title="Intensity [cps]",
    yaxis_type="log",
)
fig.show()

In [ ]:
# Gate: stop "Run All" here unless global clustering is enabled
if not run_global_clustering:

    class _Stop(Exception):
        def _render_traceback_(self):
            return [
                "\x1b[33mSection 2 skipped. Set run_global_clustering = True "
                "in the first cell to enable, or run cells below manually.\x1b[0m"
            ]

    raise _Stop()

---

## Section 2: Global peak clustering

Compute the full pairwise correlation matrix and use **hierarchical clustering**
to group peaks that behave similarly across samples.

> **Note:** This can be slow with many m/z bins. The cell below limits to the
> top-N bins by median intensity. Increase `max_bins` if your machine allows it.


In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# Limit the number of bins for performance
max_bins = 200

# Select top bins by median intensity
top_bins = intensity_matrix.median().nlargest(max_bins).index
matrix_subset = intensity_matrix[top_bins]

print(f"Computing correlation matrix for {len(top_bins)} bins...")

# Pairwise Pearson correlation (fill NaN with 0 for distance computation)
corr_matrix = matrix_subset.corr(min_periods=min_periods)

# Convert correlation to distance: d = 1 - |r|
dist_matrix = 1 - corr_matrix.abs().fillna(0)
np.fill_diagonal(dist_matrix.values, 0)

# Hierarchical clustering
condensed = squareform(dist_matrix, checks=False)
Z = linkage(condensed, method="average")

print("Done.")

In [ ]:
# Clustered heatmap
import plotly.figure_factory as ff


# Build labels for the axes
heatmap_labels = []
for b in top_bins:
    lbl = f"{bin_mz[b]:.4f}"
    if b in bin_labels.index:
        lbl += f" ({bin_labels.loc[b, 'target_ion_formula']})"
    heatmap_labels.append(lbl)

fig = ff.create_dendrogram(
    dist_matrix.values,
    orientation="bottom",
    labels=heatmap_labels,
    linkagefun=lambda x: Z,
)
fig.update_layout(
    title="Hierarchical clustering of m/z bins by correlation distance",
    xaxis_title="m/z bin",
    yaxis_title="Distance (1 − |r|)",
    height=500,
)
fig.show()

In [ ]:
# Cut the dendrogram to form flat clusters
distance_threshold = 0.3  # peaks with r > 0.7 end up in the same cluster

cluster_labels = fcluster(Z, t=distance_threshold, criterion="distance")

cluster_df = pd.DataFrame(
    {
        "mz_bin": top_bins,
        "mz": [bin_mz[b] for b in top_bins],
        "cluster": cluster_labels,
    }
)
if not bin_labels.empty:
    cluster_df = cluster_df.merge(
        bin_labels, left_on="mz_bin", right_index=True, how="left"
    )

# Show clusters with >1 member (correlated groups)
multi = cluster_df.groupby("cluster").filter(lambda g: len(g) > 1)
print(
    f"{multi['cluster'].nunique()} clusters with 2+ members ({len(multi)} bins total)"
)
multi.sort_values(["cluster", "mz"])

In [ ]:
# Plot timeseries per cluster with a dropdown to switch between them
import plotly.graph_objects as go


cluster_ids = sorted(multi["cluster"].unique())
fig = go.Figure()

# Track which traces belong to which cluster
traces_per_cluster = {}
trace_idx = 0

for cid in cluster_ids:
    c_bins = multi.loc[multi["cluster"] == cid, "mz_bin"].values
    c_ts = intensity_matrix[c_bins].copy()
    c_ts = c_ts.join(sample_dates).melt(
        id_vars="datetime_utc", var_name="mz_bin", value_name="height"
    )
    c_ts["label"] = c_ts["mz_bin"].map(_bin_label)
    c_ts = c_ts.dropna(subset=["height"])

    traces_per_cluster[cid] = []
    for label, grp in c_ts.groupby("label"):
        fig.add_trace(
            go.Scatter(
                x=grp["datetime_utc"],
                y=grp["height"],
                mode="markers",
                name=label,
                visible=(cid == cluster_ids[0]),
            )
        )
        traces_per_cluster[cid].append(trace_idx)
        trace_idx += 1

# Build dropdown buttons
buttons = []
for cid in cluster_ids:
    n_members = len(multi[multi["cluster"] == cid])
    visibility = [False] * trace_idx
    for i in traces_per_cluster[cid]:
        visibility[i] = True
    buttons.append(
        dict(
            label=f"Cluster {cid} ({n_members} bins)",
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"Cluster {cid}: co-varying signals ({n_members} bins)"},
            ],
        )
    )

fig.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            x=-0.1,
            xanchor="left",
            y=1.6,
            yanchor="top",
        )
    ],
    title=f"Cluster {cluster_ids[0]}: co-varying signals ({len(traces_per_cluster[cluster_ids[0]])} bins)",
    xaxis_title="Datetime (UTC)",
    yaxis_title="Intensity [cps]",
    yaxis_type="log",
)
fig.show()